In [1]:
import os
import soundfile as sf
from datasets import load_from_disk
import torch
import torchaudio
from models import voicecraft
import numpy as np
import random
import json
import requests
from encodec import EncodecModel
from encodec.utils import convert_audio
from hebrew import Hebrew
from hebrew.chars import HebrewChar
import pickle
from data.tokenizer import AudioTokenizer, TextTokenizer
from inference_tts_scale import inference_one_sample
import re


In [2]:
# Check the sampling rate and format

# Load directly from  local folder
dataset = load_from_disk("./fleurs_hebrew")
sample_audio = dataset[0]['audio']

# VoiceCraft usually needs 16000Hz. Let's confirm.
print(f"Current Sampling Rate: {sample_audio['sampling_rate']} Hz")

Current Sampling Rate: 16000 Hz


In [3]:
# Test if the tokenizer handles Hebrew

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model, tokenizer, and weights
voicecraft_name = "giga330M.pth"
ckpt_fn = f"./pretrained_models/{voicecraft_name}"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

ckpt = torch.load(ckpt_fn, map_location="cpu")
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
phn2num = ckpt['phn2num']

text_tokenizer = TextTokenizer(backend="espeak")
# Re-initialize the tokenizer with Hebrew support
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")
audio_tokenizer = AudioTokenizer(signature=encodec_fn) 

# Test if tokenizer can process Hebrew text
test_text = "שלום, מה נשמע?"
try:
    tokens = text_tokenizer_he(test_text)
    print(f"Text: {test_text}")
    print(f"Tokens/Phonemes: {tokens}")
except Exception as e:
    print(f"Error: {e}")

Dora directory: /tmp/audiocraft_sukiennik
/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


Text: שלום, מה נשמע?
Tokens/Phonemes: [['ʃ', 'a', 'l', 'o', 'm', ',', '_', 'm', 'a', '_', 'n', 'ʃ', 'm', 'ʔ', '?']]


In [18]:
# Checking the size of the vocabulary embedded in the checkpoint
original_vocab_size = len(ckpt['phn2num'])
print(f"The original model has {original_vocab_size} phonemes in its memory.")

# Let's see the first 5 to see if it's the English one
print("First 5 phonemes in checkpoint:", list(ckpt['phn2num'].items())[:5])

The original model has 80 phonemes in its memory.
First 5 phonemes in checkpoint: [('ɑː', 0), ('u', 1), ('aɪɚ', 2), ('ɔ', 3), ('x', 4)]


In [15]:
# hyperparameters for inference
codec_audio_sr = 16000
codec_sr = 50
top_k = 0
top_p = 0.8
temperature = 1
silence_tokens=[1388,1898,131]
kvcache = 1 # NOTE if OOM, change this to 0, or try the 330M model

# NOTE adjust the below three arguments if the generation is not as good
stop_repetition = 3 # NOTE if the model generate long silence, reduce the stop_repetition to 3, 2 or even 1
sample_batch_size = 3 # for gigaHalfLibri330M_TTSEnhanced_max16s.pth, 1 or 2 should be fine since the model is trained to do TTS, for the other two models, might need a higher number. NOTE: if the if there are long silence or unnaturally strecthed words, increase sample_batch_size to 5 or higher. What this will do to the model is that the model will run sample_batch_size examples of the same audio, and pick the one that's the shortest. So if the speech rate of the generated is too fast change it to a smaller number.
seed = 1 # change seed if you are still unhappy with the result

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed)

decode_config = {
    'top_k': top_k,
    'top_p': top_p,
    'temperature': temperature,
    'stop_repetition': stop_repetition,
    'kvcache': kvcache, 
    "codec_audio_sr": codec_audio_sr,
    "codec_sr": codec_sr, 
    "silence_tokens": silence_tokens, 
    "sample_batch_size": sample_batch_size}

In [32]:
# Running a zero-shot Hebrew inference test with the original 80-phoneme model

# Prepare the inputs for the first Hebrew generation
# Path to one of the 16kHz files we created
cut_off_sec = 4.01
audio_fn = "voicecraft_samples/sample_8.wav"
# audio_fn = "demo/Daniella.wav"

# The transcript
# original_prompt_transcript = "זו הרכישה הגדולה ביותר בתולדות ebay"
original_prompt_transcript = "טיולים הם פעילות מחוץ לבית, שכוללת הליכה בסביבה טבעית במקרים רבים בשבילי הליכה."

# The text you want the model to say in that voice
# target_transcript = "אני חושבת שתהיה תקיפה בחמישי בלילה"
# target_transcript_test = "ani khoshevet she-ti-ye tkifa be-khamishi ba-layla"
# target_transcript_test = "זו הרכישה הגדולה ביותר בתולדות "
# target_transcript_test = "This is a test to see if you can speak."
# target_transcript_test = "This is a longer test to ensure the model has enough audio data to decode properly without crashing."
# target_transcript_test = "Hello, my name is Daniella. I am currently exploring new technologies and testing how amazing this model is, i can't beleive it works"
target_transcript = "טיולים הם פעילות מחוץ לבית, שכוללת הליכה בסביבה טבעית אני חושבת שתהיה תקיפה בחמישי בלילה"


# Get frame count
info = torchaudio.info(audio_fn)
# prompt_end_frame = info.num_frames 
prompt_end_frame = int(cut_off_sec * info.sample_rate)

# Run Inference
with torch.no_grad():
    with torch.cuda.amp.autocast():
        # NOTE: Using target_transcript (Hebrew) and text_tokenizer_he
        concated_audio, gen_audio = inference_one_sample(
            model, 
            ckpt["config"], 
            phn2num,                # This is the 80-phoneme map from the model
            # text_tokenizer,  
            text_tokenizer_he,      # Hebrew G2P
            audio_tokenizer, 
            audio_fn, 
            target_transcript,     # The actual Hebrew text
            # target_transcript_test, 
            device, 
            decode_config, 
            prompt_end_frame
        )

# Post-processing for display
concated_audio, gen_audio = concated_audio[0].cpu(), gen_audio[0].cpu()

# display the audio
from IPython.display import Audio
print("Concatenate prompt and generated:")
display(Audio(concated_audio, rate=codec_audio_sr))

print("Generated Audio (Zero-shot Hebrew):")
display(Audio(gen_audio, rate=codec_audio_sr))

Concatenate prompt and generated:


Generated Audio (Zero-shot Hebrew):


In [19]:
# Comparing what the English vs Hebrew tokenizer sees
text_to_test = "זו הרכישה הגדולה ביותר"

# English Tokenizer
try:
    en_tokens = text_tokenizer(text_to_test)
    print(f"English Tokenizer sees: {en_tokens}")
except Exception as e:
    print(f"English Tokenizer Error: {e}")

# Hebrew Tokenizer
try:
    he_tokens = text_tokenizer_he(text_to_test)
    print(f"Hebrew Tokenizer sees: {he_tokens}")
except Exception as e:
    print(f"Hebrew Tokenizer Error: {e}")

English Tokenizer sees: [['h', 'iː', 'b', 'ɹ', 'uː', 'z', 'æ', 'j', 'i', 'n', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'v', 'æ', 'v', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'h', 'e', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'r', 'e', 'ʃ', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'k', 'æ', 'f', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'j', 'o', 'd', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'ʃ', 'i', 'n', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'h', 'e', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'h', 'e', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'ɡ', 'i', 'm', 'e', 'l', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'd', 'æ', 'l', 'e', 't', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'v', 'æ', 'v', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'l', 'æ', 'm', 'e', 'd', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'h', 'e', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'b', 'e', 't', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'j', 'o', 'd', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'v', 'æ', 'v', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 't', 'æ', 'v', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'r', 'e', 'ʃ']]
Hebrew Tokenizer sees: [['z', 'v', '_', 'ʔ', 'ʁ', 'χ', 'i', 'ʃ', 

In [7]:
# --- Main Manifest Creation Loop ---
base_dir = "./voicecraft_data/manifest"
os.makedirs(base_dir, exist_ok=True)
manifest_path = os.path.join(base_dir, "hebrew_manifest.jsonl")

# Directory for saving extracted WAV files
output_wav_dir = "./fleurs_hebrew/voicecraft_samples"
os.makedirs(output_wav_dir, exist_ok=True)

num_samples = len(dataset)
manifest = []

print(f"Starting to prepare manifest for {num_samples} samples...")

for i in range(num_samples):
    try:
        sample = dataset[i]
        raw_text = sample['transcription']

        # Save the audio file as WAV
        # Extracting audio array and sampling rate from the dataset
        audio_array = torch.tensor(sample['audio']['array']).unsqueeze(0)
        sr = sample['audio']['sampling_rate']

        # Calculate duration in Encodec tokens (VoiceCraft standard)
        # Encodec at 16kHz produces 50 tokens per second.
        # Formula: (num_samples / sampling_rate) * 50
        num_audio_samples = audio_array.shape[1]
        duration_frames = int((num_audio_samples / sr) * 50)
        
        wav_filename = f"sample_{i}.wav"
        wav_path = os.path.join(output_wav_dir, wav_filename)
        
        # Physical save to disk using torchaudio
        torchaudio.save(wav_path, audio_array, sr)
    
        # Get Phonemes
        phonemes_list = text_tokenizer_he(raw_text)[0]
        phonemes_str = " ".join(phonemes_list)
        
        # Append to manifest with relative path for VoiceCraft
        manifest.append({
            "audio_filepath": f"{output_wav_dir}/{wav_filename}",
            "text": raw_text,
            "phonemes": phonemes_str,
            "duration_frames": duration_frames
        })

        # Print sample 0 to verify everything looks correct
        if i == 0:
            print(f"--- DEBUG Sample 0 ---")
            print(f"Text: {raw_text}")
            print(f"Phonemes: {phonemes_str[:50]}...")
            print(f"Duration: {duration_frames} frames")
            print(f"----------------------")
        
        if i % 100 == 0: print(f"Processed {i} samples...")
            
    except Exception as e:
        print(f"Error in sample {i}: {e}")

# Save the manifest
with open(manifest_path, "w", encoding="utf-8") as f:
    for entry in manifest:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("Done! Check 'hebrew_manifest.jsonl'")

Starting to prepare manifest for 3242 samples...
--- DEBUG Sample 0 ---
Text: זו הרכישה הגדולה ביותר בתולדות ebay
Phonemes: z v _ ʔ ʁ χ i ʃ ʔ _ ʔ ɡ d v l ʔ _ v i v t ʁ _ v t ...
Duration: 189 frames
----------------------
Processed 0 samples...
Processed 100 samples...
Processed 200 samples...
Processed 300 samples...
Processed 400 samples...
Processed 500 samples...
Processed 600 samples...
Processed 700 samples...
Processed 800 samples...
Processed 900 samples...
Processed 1000 samples...
Processed 1100 samples...
Processed 1200 samples...
Processed 1300 samples...
Processed 1400 samples...
Processed 1500 samples...
Processed 1600 samples...
Processed 1700 samples...
Processed 1800 samples...
Processed 1900 samples...
Processed 2000 samples...
Processed 2100 samples...
Processed 2200 samples...
Processed 2300 samples...
Processed 2400 samples...
Processed 2500 samples...
Processed 2600 samples...
Processed 2700 samples...
Processed 2800 samples...
Processed 2900 samples...
Processed 

In [21]:
# Actually expand the vocabulary and see the new phonemes
def expand_vocabulary(manifest_path, original_phn2num):
    # Create a copy so we don't overwrite the original accidentally
    new_phn2num = original_phn2num.copy()
    
    # Start numbering from the next available index
    next_index = max(original_phn2num.values()) + 1
    
    added_phonemes = []
    with open(manifest_path, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            # Split the phonemes string back into a list
            phonemes = entry['phonemes'].split()
            for p in phonemes:
                if p not in new_phn2num:
                    new_phn2num[p] = next_index
                    added_phonemes.append(p)
                    next_index += 1
    
    print(f"--- Vocabulary Expansion Complete ---")
    print(f"Added {len(added_phonemes)} new phonemes.")
    print(f"New phonemes examples: {added_phonemes[:10]}")
    return new_phn2num

# Run it!
# Expand vocabulary and save to pkl
new_phn2num = expand_vocabulary("./voicecraft_data/manifest/hebrew_manifest.jsonl", phn2num)

with open("new_phn2num.pkl", "wb") as f:
    pickle.dump(new_phn2num, f)

print(f"Vocabulary expanded. Total phonemes now: {len(new_phn2num)}")

--- Vocabulary Expansion Complete ---
Added 14 new phonemes.
New phonemes examples: ['ʁ', 'χ', '(', 'en', ')', 'he', 'e', 'a', 'o', 'əʊ']
Vocabulary expanded. Total phonemes now: 94


In [22]:
# Creating a text vocab file from the binary mapping to ensure consistency

base_dir = "./voicecraft_data"
os.makedirs(base_dir, exist_ok=True)
vocab_path = os.path.join(base_dir, "vocab.txt")

# Sorting by the index (the value in our dictionary) to match the expected order
sorted_vocab = sorted(new_phn2num.items(), key=lambda x: x[1])

with open(vocab_path, "w", encoding="utf-8") as f:
    for phn, idx in sorted_vocab:
        # We write ONLY the phoneme. The line number acts as the index.
        f.write(f"{idx} {phn}\n")
        
print(f"Success! Vocab file created at: {vocab_path}")
print(f"Vocab file created with {len(sorted_vocab)} entries.")

Success! Vocab file created at: ./voicecraft_data/vocab.txt
Vocab file created with 94 entries.


In [23]:
# Creating mini-manifests for a quick sanity check (100 train, 20 val)

base_dir = "./voicecraft_data/manifest"
os.makedirs(base_dir, exist_ok=True)
# train_path = os.path.join(base_dir, "hebrew_train_100.jsonl")
# val_path = os.path.join(base_dir, "hebrew_val_20.jsonl")

# train_mini = manifest[:100]
# val_mini = manifest[100:120]

train_path = os.path.join(base_dir, "hebrew_train_200.jsonl")
val_path = os.path.join(base_dir, "hebrew_val_40.jsonl")

train_mini = manifest[:200]
val_mini = manifest[200:220]

def save_manifest(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

save_manifest(train_mini, train_path)
save_manifest(val_mini, val_path)

print("Mini-manifests are ready!")

Mini-manifests are ready!


In [24]:
def convert_jsonl_to_voicecraft_txt(jsonl_input_path, txt_output_path, phonemes_base_dir):
    """
    Converts a JSONL manifest into the specific TXT format required by VoiceCraft
    and creates individual phoneme files for each sample.
    
    jsonl_input_path: Path to the source .jsonl file
    txt_output_path: Path where the final .txt manifest will be saved
    phonemes_base_dir: Directory where individual .txt phoneme files will be created
    """
    # Create phonemes directory if it doesn't exist
    os.makedirs(phonemes_base_dir, exist_ok=True)
    
    print(f"Processing: {jsonl_input_path} -> {txt_output_path}")
    
    samples_processed = 0
    with open(jsonl_input_path, 'r', encoding='utf-8') as f_in, \
         open(txt_output_path, 'w', encoding='utf-8') as f_out:
        
        for i, line in enumerate(f_in):
            data = json.loads(line)
            
            # Extract ID from audio_filepath (e.g., 'sample_0')
            item_id = os.path.basename(data['audio_filepath']).replace(".wav", "")
            duration = data.get('duration_frames', 0)
            phonemes = data.get('phonemes', "")
            
            # Create the individual phoneme file (e.g., ./phonemes/sample_0.txt)
            phn_file_path = os.path.join(phonemes_base_dir, f"{item_id}.txt")
            with open(phn_file_path, "w", encoding="utf-8") as f_phn:
                f_phn.write(phonemes)
            
            # Write to the manifest .txt (Format: Index <TAB> ID <TAB> Duration)
            f_out.write(f"{i}\t{item_id}\t{duration}\n")
            samples_processed += 1
            
    print(f"Successfully converted {samples_processed} samples.")

# --- Usage Example ---
# Define paths and call the function for train and validation
base_manifest_dir = "./voicecraft_data/manifest"
phn_dir = "./voicecraft_data/phonemes"

# # For Train
# convert_jsonl_to_voicecraft_txt(
#     jsonl_input_path=os.path.join(base_manifest_dir, "hebrew_train_100.jsonl"),
#     txt_output_path=os.path.join(base_manifest_dir, "train.txt"),
#     phonemes_base_dir=phn_dir
# )

# # For Validation
# convert_jsonl_to_voicecraft_txt(
#     jsonl_input_path=os.path.join(base_manifest_dir, "hebrew_val_20.jsonl"),
#     txt_output_path=os.path.join(base_manifest_dir, "validation.txt"),
#     phonemes_base_dir=phn_dir
# )

# For Train
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "hebrew_train_200.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "train.txt"),
    phonemes_base_dir=phn_dir
)

# For Validation
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "hebrew_val_40.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "validation.txt"),
    phonemes_base_dir=phn_dir
)

Processing: ./voicecraft_data/manifest/hebrew_train_200.jsonl -> ./voicecraft_data/manifest/train.txt
Successfully converted 200 samples.
Processing: ./voicecraft_data/manifest/hebrew_val_40.jsonl -> ./voicecraft_data/manifest/validation.txt
Successfully converted 20 samples.


In [ ]:
# ==========================================
# STEP: Audio Tokenization - Encoding for Training (16kHz, 4 Codebooks)
# Goal: Convert .wav files to VoiceCraft-compatible .pth tokens
#       Ensure all .wav files are encoded with the 4cb2048_giga model
# ==========================================

# Set device to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the specific VoiceCraft Encodec model (Gigaspeech 4-codebook version)
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"
audio_tokenizer = AudioTokenizer(signature=encodec_fn) 

print(f"✅ Loaded VoiceCraft AudioTokenizer from {encodec_fn}")

# Directory setup
wav_dir = "./fleurs_hebrew/voicecraft_samples"
# wav_dir = "./fleurs_hebrew/new"

save_dir = "./voicecraft_data/encodec_16khz_4codebooks"
# save_dir = "./voicecraft_data/encodec_16khz_4codebooks_new"

os.makedirs(save_dir, exist_ok=True)

# List and sort files for consistency
wav_files = sorted([f for f in os.listdir(wav_dir) if f.endswith(".wav")])

print(f"🚀 Processing {len(wav_files)} files to {save_dir}...")

# Process each wav file
for wav_file in wav_files:
    file_id = wav_file.replace(".wav", "")
    wav_path = os.path.join(wav_dir, wav_file)
    
    # Load audio (returns CPU tensor by default)
    wav, sr = torchaudio.load(wav_path)

    # Ensure mono and correct sample rate
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    
    # VoiceCraft expects 16khz mono - Resample to 16kHz on CPU (more stable)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)

    # Encode the audio to tokens
    with torch.no_grad():
        # VoiceCraft/Encodec expects [B, C, T] -> [1, 1, T]
        # Use unsqueeze(0) to turn [1, T] into [1, 1, T]
        # Input shape: [1, 1, T], Output: [(codes, scale)]
        encoded_frames = audio_tokenizer.encode(wav.unsqueeze(0).to(device))

        # Extract the codebook indices (tokens)
        # Result shape should be [num_codebooks, sequence_length]
        codes = encoded_frames[0][0].squeeze(0)            
        
    # Save as .pth file
    # Always move back to CPU before saving to avoid issues
    torch.save(codes.cpu(), os.path.join(save_dir, f"{file_id}.pth"))
        

print("Encoding finished! All .pth files are ready.")

✅ Loaded VoiceCraft AudioTokenizer from ./pretrained_models/encodec_4cb2048_giga.th
🚀 Processing 8 files to ./voicecraft_data/encodec_16khz_4codebooks_new...
Encoding finished! All .pth files are ready.


In [50]:
# ==========================================
# STEP: Validation - Checking Encodec Tokens Shape
# Goal: Verify that .pth files have 4 codebooks [4, T]
# ==========================================

# Define the directory to check
check_dir = "./voicecraft_data/encodec_16khz_4codebooks"

if os.path.exists(check_dir) and len(os.listdir(check_dir)) > 0:
    # Path to one of your generated .pth files
    sample_file = os.path.join(check_dir, os.listdir(check_dir)[0])

    # Load the tokens
    tokens = torch.load(sample_file)

    # Check the shape
    # Expected shape for VoiceCraft: [num_codebooks, sequence_length]
    num_codebooks, seq_len = tokens.shape

    print(f"File: {sample_file}")
    print(f"Number of codebooks: {num_codebooks}")
    print(f"Sequence length (frames): {seq_len}")

    if num_codebooks == 4:
        print("✅ Success! The file has 4 codebooks as expected for VoiceCraft.")
    else:
        print(f"❌ Warning: Found {num_codebooks} codebooks. VoiceCraft usually needs 4.")
else:
    print("❌ Error: Directory is empty or does not exist. Run the encoding cell first.")

File: ./voicecraft_data/encodec_16khz_4codebooks/sample_1.pth
Number of codebooks: 4
Sequence length (frames): 681
✅ Success! The file has 4 codebooks as expected for VoiceCraft.


In [60]:
# ==========================================
# STEP: Reconstruction Check (Precision Match)
# Goal: Verify encoding quality by matching tokens to their source WAV
# ==========================================

import IPython.display as ipd

# Select a specific file ID to test (e.g., the first one in the sorted list)
sample_id = sorted([f.replace(".pth", "") for f in os.listdir(save_dir) if f.endswith(".pth")])[737]

pth_file = os.path.join(save_dir, f"{sample_id}.pth")
wav_file = os.path.join(wav_dir, f"{sample_id}.wav")

print(f"Testing Consistency for ID: {sample_id}")

# Load Tokens
tokens = torch.load(pth_file).to(device)
print(f"Tokens shape: {tokens.shape}")

# Decode back to audio
with torch.no_grad():
    # Adding batch dim [1, 4, T] and formatting for AudioTokenizer
    decoded_audio = audio_tokenizer.decode([(tokens.unsqueeze(0), None)])

# Compare Playback
print("--- 🔊 Reconstructed from Tokens ---")
ipd.display(ipd.Audio(decoded_audio.squeeze().cpu().numpy(), rate=16000))

print("--- 🔈 Original Source File ---")
ipd.display(ipd.Audio(wav_file))

Testing Consistency for ID: sample_1661
Tokens shape: torch.Size([4, 429])
--- 🔊 Reconstructed from Tokens ---


--- 🔈 Original Source File ---


In [ ]:
# ==========================================
# STEP: Convert .pth Tokens to .txt for Training
# Goal: Ensure VoiceCraft DataLoader can find and read audio tokens
# ==========================================

# Define the directory containing your .pth files
encodec_dir = "./voicecraft_data/encodec_16khz_4codebooks"
# encodec_dir = "./voicecraft_data/encodec_16khz_4codebooks_new"

pth_files = [f for f in os.listdir(encodec_dir) if f.endswith(".pth")]

print(f"🔄 Converting {len(pth_files)} files from .pth to 4-row .txt...")

for f in pth_files:
    # Load the binary PyTorch file
    file_path = os.path.join(encodec_dir, f)
    # codes = torch.load(file_path) # Expected shape: [4, T]
    codes = torch.load(file_path).cpu().numpy() # Expected shape: [4, T]
    
    # Define the new .txt path
    txt_path = file_path.replace(".pth", ".txt")

    # Save as text where each codebook is on a NEW line
    # This is often required by VoiceCraft's np.loadtxt calls
    np.savetxt(txt_path, codes, fmt='%d')
    
    # Flatten the 4 codebooks into a single sequence and save as space-separated text
    # VoiceCraft's Gigaspeech loader typically expects space-separated integers
    # flat_codes = codes.view(-1).cpu().numpy().tolist()
    
    # with open(txt_path, 'w') as tf:
    #     tf.write(" ".join(map(str, flat_codes)))

print("✅ Conversion complete. All tokens are now available as .txt files.")

🔄 Converting 8 files from .pth to 4-row .txt...
✅ Conversion complete. All tokens are now available as .txt files.


In [6]:
# ==========================================
# STEP: Verify .txt Token Format
# Goal: Ensure the text file is correctly formatted for VoiceCraft
# ==========================================


# Pick the first .txt file generated
txt_dir = "./voicecraft_data/encodec_16khz_4codebooks"
txt_files = sorted([f for f in os.listdir(txt_dir) if f.endswith(".txt")])

if not txt_files:
    print("❌ No .txt files found! Run the conversion step first.")
else:
    sample_txt = os.path.join(txt_dir, txt_files[0])
    
    # 2. Read the content
    with open(sample_txt, 'r') as f:
        content = f.read().strip()
    
    # 3. Split by space to get individual tokens
    tokens = content.split()
    
    print(f"📄 Checking file: {sample_txt}")
    print(f"🔢 Total tokens found: {len(tokens)}")
    print(f"💡 First 10 tokens: {tokens[:10]}")
    
    # 4. Validation Checks
    is_numeric = all(t.isdigit() for t in tokens)
    is_multiple_of_4 = len(tokens) % 4 == 0
    
    if is_numeric and is_multiple_of_4:
        print("✅ Format looks perfect: All tokens are integers and divisible by 4.")
    elif not is_numeric:
        print("❌ Error: Found non-numeric characters in the file.")
    elif not is_multiple_of_4:
        print("⚠️ Warning: Token count is not a multiple of 4. Check if flattening was correct.")

📄 Checking file: ./voicecraft_data/encodec_16khz_4codebooks/sample_0.txt
🔢 Total tokens found: 756
💡 First 10 tokens: ['1898', '1388', '1388', '1388', '1388', '1388', '1898', '131', '1174', '1620']
✅ Format looks perfect: All tokens are integers and divisible by 4.


In [10]:
# ==========================================
# STEP: Verify Token Order and Reconstruction
# Goal: Ensure the .txt format matches VoiceCraft's expectation [4, T]
# ==========================================

import IPython.display as ipd

save_dir = "./voicecraft_data/encodec_16khz_4codebooks"
wav_dir = "./fleurs_hebrew/voicecraft_samples"

sample_id = sorted([f.replace(".pth", "") for f in os.listdir(save_dir) if f.endswith(".pth")])[1]
wav_file = os.path.join(wav_dir, f"{sample_id}.wav")

# Load the text file you just created
sample_txt = "./voicecraft_data/encodec_16khz_4codebooks/sample_1.txt" # Change to your actual filename
with open(sample_txt, 'r') as f:
    flat_tokens = list(map(int, f.read().split()))

# Reshape back to [4, T] 
# VoiceCraft expects the flat list to be divisible by 4
num_tokens = len(flat_tokens)
T = num_tokens // 4
tokens_reshaped = torch.tensor(flat_tokens).view(4, T).to(device)

print(f"✅ Loaded {num_tokens} tokens, reshaped to: {tokens_reshaped.shape}")

# Decode back to audio to verify quality
with torch.no_grad():
    # Wrap in the list/tuple format required by AudioTokenizer
    # We add batch dim [1, 4, T]
    reconstructed_audio = audio_tokenizer.decode([(tokens_reshaped.unsqueeze(0), None)])

# Listen - If it sounds like the original, the order is 100% correct
print("--- 🔊 Testing Reconstruction from .txt file ---")
ipd.display(ipd.Audio(reconstructed_audio.squeeze().cpu().numpy(), rate=16000))

print("--- 🔈 Original Source File ---")
ipd.display(ipd.Audio(wav_file))

✅ Loaded 2724 tokens, reshaped to: torch.Size([4, 681])
--- 🔊 Testing Reconstruction from .txt file ---


--- 🔈 Original Source File ---


In [ ]:
# train 100 val 20 
# 500 steps
!bash ./z_scripts/e330.sh

[2026-03-01 23:05:33,837] torch.distributed.run: [WARNING] master_addr is only used for static rdzv_backend and when rdzv_endpoint is not specified.
2026-03-01 23:05:36,137 [INFO] main.py:19 || Namespace(seed=1, precision='float16', num_workers=2, resume=False, tb_write_every_n_steps=100, print_every_n_steps=10, val_every_n_steps=25, lr=0.0005, batch_size=1, max_num_tokens=100000, val_max_num_tokens=None, num_buckets=6, dynamic_batching=0, weight_decay=0.01, warmup_fraction=0.01, num_epochs=10, num_steps=500, gradient_accumulation_steps=8, gradient_clip_val=1.0, early_stop_step=3200, early_stop_threshold=-1.0, optimizer_name='AdamW', reduce_lr_start_step=3000, pseudo_epoch_size=3000, reduce_lr_start_epoch=4, clipping_update_period=600, exp_dir='./experiments/hebrew_fleurs/hebrew_v1_330M', dataset='gigaspeech', dataset_dir='./voicecraft_data', phn_folder_name='phonemes', encodec_folder_name='encodec_16khz_4codebooks', manifest_name='manifest', pad_x=1, audio_max_length=20, audio_min_len

In [ ]:
# train 3 val 3
# 500 steps
!bash ./z_scripts/e330.sh

[2026-03-02 14:51:14,723] torch.distributed.run: [WARNING] master_addr is only used for static rdzv_backend and when rdzv_endpoint is not specified.
2026-03-02 14:51:20,013 [INFO] main.py:19 || Namespace(seed=1, precision='float16', num_workers=2, resume=False, tb_write_every_n_steps=100, print_every_n_steps=10, val_every_n_steps=25, lr=0.0005, batch_size=1, max_num_tokens=100000, val_max_num_tokens=None, num_buckets=6, dynamic_batching=0, weight_decay=0.01, warmup_fraction=0.01, num_epochs=10, num_steps=500, gradient_accumulation_steps=8, gradient_clip_val=1.0, early_stop_step=3200, early_stop_threshold=-1.0, optimizer_name='AdamW', reduce_lr_start_step=3000, pseudo_epoch_size=3000, reduce_lr_start_epoch=4, clipping_update_period=600, exp_dir='./experiments/hebrew_fleurs/hebrew_v1_330M_220', dataset='gigaspeech', dataset_dir='./voicecraft_data', phn_folder_name='phonemes', encodec_folder_name='encodec_16khz_4codebooks', manifest_name='manifest', pad_x=1, audio_max_length=20, audio_min

In [4]:
# Test if the tokenizer handles Hebrew

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model, tokenizer, and weights
# voicecraft_name = "giga330M.pth"
# ckpt_fn = f"./pretrained_models/{voicecraft_name}"
ckpt_fn = "experiments/hebrew_fleurs/hebrew_v1_330M_220/best_bundle.pth"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

ckpt = torch.load(ckpt_fn, map_location="cpu")
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
phn2num = ckpt['phn2num']

# Re-initialize the tokenizer with Hebrew support
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")
audio_tokenizer = AudioTokenizer(signature=encodec_fn) 

Dora directory: /tmp/audiocraft_sukiennik
/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [13]:
# hyperparameters for inference
codec_audio_sr = 16000
codec_sr = 50
top_k = 0
top_p = 0.5
temperature = 1
silence_tokens=[1388,1898,131]
kvcache = 1 # NOTE if OOM, change this to 0, or try the 330M model

# NOTE adjust the below three arguments if the generation is not as good
stop_repetition = 1 # NOTE if the model generate long silence, reduce the stop_repetition to 3, 2 or even 1
sample_batch_size = 3 # for gigaHalfLibri330M_TTSEnhanced_max16s.pth, 1 or 2 should be fine since the model is trained to do TTS, for the other two models, might need a higher number. NOTE: if the if there are long silence or unnaturally strecthed words, increase sample_batch_size to 5 or higher. What this will do to the model is that the model will run sample_batch_size examples of the same audio, and pick the one that's the shortest. So if the speech rate of the generated is too fast change it to a smaller number.
seed = 1 # change seed if you are still unhappy with the result

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed)

decode_config = {
    'top_k': top_k,
    'top_p': top_p,
    'temperature': temperature,
    'stop_repetition': stop_repetition,
    'kvcache': kvcache, 
    "codec_audio_sr": codec_audio_sr,
    "codec_sr": codec_sr, 
    "silence_tokens": silence_tokens, 
    "sample_batch_size": sample_batch_size}

In [14]:
# Running a zero-shot Hebrew inference test with the original 80-phoneme model

# Prepare the inputs for the first Hebrew generation
# Path to one of the 16kHz files we created
cut_off_sec = 4.01
audio_fn = "fleurs_hebrew/voicecraft_samples/sample_220.wav"
# audio_fn = "demo/Daniella.wav"

# The transcript
# original_prompt_transcript = "זו הרכישה הגדולה ביותר בתולדות ebay"
# original_prompt_transcript = "טיולים הם פעילות מחוץ לבית, שכוללת הליכה בסביבה טבעית במקרים רבים בשבילי הליכה."

# The text you want the model to say in that voice
# target_transcript = "אני חושבת שתהיה תקיפה בחמישי בלילה"
# target_transcript_test = "ani khoshevet she-ti-ye tkifa be-khamishi ba-layla"
# target_transcript_test = "זו הרכישה הגדולה ביותר בתולדות "
# target_transcript_test = "This is a test to see if you can speak."
# target_transcript_test = "This is a longer test to ensure the model has enough audio data to decode properly without crashing."
# target_transcript_test = "Hello, my name is Daniella. I am currently exploring new technologies and testing how amazing this model is, i can't beleive it works"
# target_transcript = "טיולים הם פעילות מחוץ לבית, שכוללת הליכה בסביבה טבעית אני חושבת שתהיה תקיפה בחמישי בלילה"
# target_transcript = "גם גולף וגם רוגבי אמורים לחזור למשחקים האולימפיים"
target_transcript = "ועורכי דין כלליים ומומחים במהלך השנים הקודמות"

# Get frame count
info = torchaudio.info(audio_fn)
# prompt_end_frame = info.num_frames 
prompt_end_frame = int(cut_off_sec * info.sample_rate)

# Run Inference
with torch.no_grad():
    with torch.cuda.amp.autocast():
        # NOTE: Using target_transcript (Hebrew) and text_tokenizer_he
        concated_audio, gen_audio = inference_one_sample(
            model, 
            ckpt["config"], 
            phn2num,                # This is the 80-phoneme map from the model
            # text_tokenizer,  
            text_tokenizer_he,      # Hebrew G2P
            audio_tokenizer, 
            audio_fn, 
            target_transcript,     # The actual Hebrew text
            # target_transcript_test, 
            device, 
            decode_config, 
            prompt_end_frame
        )

# Post-processing for display
concated_audio, gen_audio = concated_audio[0].cpu(), gen_audio[0].cpu()

# display the audio
from IPython.display import Audio
print("Concatenate prompt and generated:")
display(Audio(concated_audio, rate=codec_audio_sr))

print("Generated Audio (Zero-shot Hebrew):")
display(Audio(gen_audio, rate=codec_audio_sr))

Concatenate prompt and generated:


Generated Audio (Zero-shot Hebrew):


In [7]:
# Function to check which layers are trainable
def check_trainable_parameters(model):
    print("--- Trainable Layers Status ---")
    for name, param in model.named_parameters():
        # Check if the parameter is set to be updated
        status = "TRAINABLE" if param.requires_grad else "FROZEN"
        print(f"{name:50} | {status}")

# Run the check
check_trainable_parameters(model)

--- Trainable Layers Status ---
eog                                                | FROZEN
eos                                                | FROZEN
mask_embedding                                     | TRAINABLE
text_embedding.word_embeddings.weight              | TRAINABLE
audio_embedding.0.word_embeddings.weight           | TRAINABLE
audio_embedding.1.word_embeddings.weight           | TRAINABLE
audio_embedding.2.word_embeddings.weight           | TRAINABLE
audio_embedding.3.word_embeddings.weight           | TRAINABLE
text_positional_embedding.alpha                    | TRAINABLE
audio_positional_embedding.alpha                   | TRAINABLE
decoder.layers.0.self_attn.in_proj_weight          | TRAINABLE
decoder.layers.0.self_attn.in_proj_bias            | TRAINABLE
decoder.layers.0.self_attn.out_proj.weight         | TRAINABLE
decoder.layers.0.self_attn.out_proj.bias           | TRAINABLE
decoder.layers.0.linear1.weight                    | TRAINABLE
decoder.layers.0.linear1.bias

In [10]:
# English comment: Compare current phonemes with the model's dictionary
text_tokenizer = TextTokenizer(backend="espeak")

phonemes = text_tokenizer_he(target_transcript)
print(f"Generated Phonemes: {phonemes}")

phonemes_en = text_tokenizer(target_transcript)
print(f"Generated Phonemes english: {phonemes_en}")

# English comment: Flatten the list of lists to a single list of phonemes
flat_phonemes = [p for sentence in phonemes for p in sentence]

# Now check for missing phonemes
missing = [p for p in flat_phonemes if p not in phn2num]
print(f"Phonemes missing from model dictionary: {missing}")

# If you want to see the clean list
print(f"Flat phonemes: {flat_phonemes}") 


Generated Phonemes: [['v', 'ʔ', 'v', 'ʁ', 'χ', 'i', '_', 'd', 'i', 'n', '_', 'χ', 'l', 'l', 'i', 'j', 'm', '_', 'v', 'm', 'v', 'm', 'χ', 'i', 'm', '_', 'v', 'm', 'ʔ', 'l', 'χ', '_', 'ʔ', 'ʃ', 'n', 'i', 'm', '_', 'ʔ', 'k', 'v', 'd', 'm', 'v', 't']]
Generated Phonemes english: [['h', 'iː', 'b', 'ɹ', 'uː', 'v', 'æ', 'v', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'ʔ', 'æ', 'j', 'i', 'n', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'v', 'æ', 'v', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'r', 'e', 'ʃ', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'k', 'æ', 'f', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'j', 'o', 'd', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'd', 'æ', 'l', 'e', 't', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'j', 'o', 'd', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'n', 'u', 'n', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'k', 'æ', 'f', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'l', 'æ', 'm', 'e', 'd', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'l', 'æ', 'm', 'e', 'd', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'j', 'o', 'd', '_', 'h', 'iː', 'b', 'ɹ', 'uː', 'j', 'o', 'd', '_', 'h', 'iː', 'b', 'ɹ', 'uː

In [16]:
# Check if the phonemes in your list actually exist in the model's vocabulary
current_phonemes = ['ɡ', 'm', '_', 'ɡ', 'v', 'l', 'f', 'ʁ'] # common ones from your list
for p in current_phonemes:
    if p in phn2num:
        print(f"Phoneme '{p}' is OK (Index: {phn2num[p]})")
    else:
        print(f"!!! Phoneme '{p}' is MISSING from phn2num !!!")

Phoneme 'ɡ' is OK (Index: 49)
Phoneme 'm' is OK (Index: 9)
Phoneme '_' is OK (Index: 15)
Phoneme 'ɡ' is OK (Index: 49)
Phoneme 'v' is OK (Index: 31)
Phoneme 'l' is OK (Index: 63)
Phoneme 'f' is OK (Index: 69)
Phoneme 'ʁ' is OK (Index: 80)


In [26]:
# English comment: Print all attributes in the config to find the correct key
print(vars(ckpt['config']))

{'seed': 1, 'precision': 'float16', 'num_workers': 2, 'resume': False, 'tb_write_every_n_steps': 100, 'print_every_n_steps': 10, 'val_every_n_steps': 25, 'lr': 0.0005, 'batch_size': 1, 'max_num_tokens': 100000, 'val_max_num_tokens': None, 'num_buckets': 6, 'dynamic_batching': 0, 'weight_decay': 0.01, 'warmup_fraction': 0.01, 'num_epochs': 5, 'num_steps': 500, 'gradient_accumulation_steps': 8, 'gradient_clip_val': 1.0, 'early_stop_step': 3200, 'early_stop_threshold': -1.0, 'optimizer_name': 'AdamW', 'reduce_lr_start_step': 3000, 'pseudo_epoch_size': 3000, 'reduce_lr_start_epoch': 4, 'clipping_update_period': 600, 'exp_dir': './experiments/hebrew_fleurs/hebrew_v1_330M', 'dataset': 'gigaspeech', 'dataset_dir': './voicecraft_data', 'phn_folder_name': 'phonemes', 'encodec_folder_name': 'encodec_16khz_4codebooks', 'manifest_name': 'manifest', 'pad_x': 1, 'audio_max_length': 20, 'audio_min_length': 2, 'text_max_length': 400, 'text_min_length': 10, 'encodec_sr': 50, 'drop_long': 0, 'mask_len_m

In [ ]:
היי אני מנסה לשנות את voicecraft שיתאים לעברית, הרצתי אימון קטן על 100 דוגמאות ו-20 בדיקות, אני לא מקבלת תוצאות טובות וזה מובן, אבל לפני שאני מריצה אימון גדול ני רוצה לראות שאני בכיוון. אני מנסה לייצר קול של משפט שקיים לי בקבצי האימון ולא מקבלת תוצאה טובה (מקבלת רעש) מה יכולה להיות הבעיה?

In [4]:
# train 200 val 40
# 2000 steps
# !bash ./z_scripts/e330.sh
!bash ./z_scripts/e330.sh 2>&1 | tee experiments/hebrew_fleurs/hebrew_v1_330M_200/training_log.txt

tee: experiments/hebrew_fleurs/hebrew_v1_330M_200/training_log.txt: No such file or directory
[2026-03-02 21:47:58,767] torch.distributed.run: [WARNING] master_addr is only used for static rdzv_backend and when rdzv_endpoint is not specified.
2026-03-02 21:48:00,463 [INFO] main.py:19 || Namespace(seed=1, precision='float16', num_workers=2, resume=False, tb_write_every_n_steps=100, print_every_n_steps=10, val_every_n_steps=50, lr=1e-05, batch_size=1, max_num_tokens=100000, val_max_num_tokens=None, num_buckets=6, dynamic_batching=0, weight_decay=0.01, warmup_fraction=0.1, num_epochs=10, num_steps=2000, gradient_accumulation_steps=8, gradient_clip_val=1.0, early_stop_step=200, early_stop_threshold=0.001, optimizer_name='AdamW', reduce_lr_start_step=3000, pseudo_epoch_size=3000, reduce_lr_start_epoch=4, clipping_update_period=600, exp_dir='./experiments/hebrew_fleurs/hebrew_v1_330M_200', dataset='gigaspeech', dataset_dir='./voicecraft_data', phn_folder_name='phonemes', encodec_folder_name=

In [50]:
# faster for fine tunning (Fine-tuning from GigaSpeech)
# Running with frequent saving and optimized batching
!torchrun --nproc_per_node=1 main.py \
--dataset_dir "./voicecraft_data" \
--exp_dir "./experiments/hebrew_fleurs/hebrew_v1_330M" \
--dataset "gigaspeech" \
--manifest_name "manifest" \
--text_vocab_size 100 \
--text_pad_token 100 \
--num_steps 500 \
--lr 0.0001 \
--batch_size 2 \
--gradient_accumulation_steps 2 \
--n_codebooks 4 \
--audio_vocab_size 2048 \
--num_workers 4 \
--print_every_n_steps 1 \
--load_model_from "./pretrained_models/giga330M_fixed.pth" \
--val_every_n_steps 25  # <--- Saving and validat every 25 steps so we never lose more than 30 mins of work

2026-03-01 02:14:03,949 [INFO] main.py:19 || Namespace(seed=1, precision='float16', num_workers=4, resume=False, tb_write_every_n_steps=100, print_every_n_steps=1, val_every_n_steps=25, lr=0.0001, batch_size=2, max_num_tokens=100000, val_max_num_tokens=None, num_buckets=6, dynamic_batching=0, weight_decay=0.01, warmup_fraction=0.01, num_epochs=10, num_steps=500, gradient_accumulation_steps=2, gradient_clip_val=1.0, early_stop_step=3200, early_stop_threshold=-1.0, optimizer_name='AdamW', reduce_lr_start_step=3000, pseudo_epoch_size=3000, reduce_lr_start_epoch=4, clipping_update_period=600, exp_dir='./experiments/hebrew_fleurs/hebrew_v1_330M', dataset='gigaspeech', dataset_dir='./voicecraft_data', phn_folder_name='phonemes', encodec_folder_name='encodec_16khz_4codebooks', manifest_name='manifest', pad_x=1, audio_max_length=20, audio_min_length=2, text_max_length=400, text_min_length=10, encodec_sr=50, drop_long=0, mask_len_min=1, mask_len_max=600, eos=-1, reduced_eog=0, special_first=0, 

In [49]:
ckpt_path = "./pretrained_models/giga330M.pth"
fixed_path = "./pretrained_models/giga330M_fixed.pth"

# 1. Load the original file
sd = torch.load(original_ckpt, map_location="cpu")

# 2. Extract weights (handle different save formats)
weights = sd['model'] if 'model' in sd else sd

# 3. Clean the keys
cleaned_weights = {k.replace("module.", "").replace("model.", ""): v for k, v in weights.items()}

# 2. Wrap them back in a 'model' dictionary so trainer.py line 397 is happy
final_sd = {'model': cleaned_weights}

torch.save(final_sd, fixed_path)
print(f"Fixed and wrapped checkpoint saved to {fixed_path}.")

Fixed and wrapped checkpoint saved to ./pretrained_models/giga330M_fixed.pth.


In [42]:
# English comments: Inspecting the checkpoint to see the actual vocabulary size
import torch

checkpoint_path = "./pretrained_models/giga330M.pth"
sd = torch.load(checkpoint_path, map_location="cpu")

# Getting the shape of the text embedding layer
if 'model' in sd:
    text_emb_shape = sd['model']['text_embedding.word_embeddings.weight'].shape
    print(f"Checkpoint text embedding shape: {text_emb_shape}")
    # The first number in the shape is the vocab size
else:
    print("Could not find 'model' key in checkpoint")

Checkpoint text embedding shape: torch.Size([101, 2048])


In [45]:
# English comments: Listing all attributes of the tokenizer to find the dictionary name
from data.tokenizer import TextTokenizer
t_tokenizer = TextTokenizer()

print("Available attributes in your tokenizer:")
print(dir(t_tokenizer))

# Let's also try to see if it has something like 'phoneme2id'
if hasattr(t_tokenizer, 'phoneme2id'):
    print(f"Found it! Phoneme2id size: {len(t_tokenizer.phoneme2id)}")

Available attributes in your tokenizer:
['__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'backend', 'separator', 'to_list']


In [6]:
# Path configuration
base_dir = "./voicecraft_data"
manifest_path = os.path.join(base_dir, "manifest/train.txt")
phonemes_dir = os.path.join(base_dir, "phonemes")
os.makedirs(phonemes_dir, exist_ok=True)

print("Starting Manifest and Phonemes cleanup...")

fixed_lines = []
if os.path.exists(manifest_path):
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 3:
                # Column 0: ID, Column 1: Speaker (setting to ID), Column 2: Phonemes, Column 3: Duration
                file_id = parts[0]
                phonemes = parts[2]
                duration = parts[3] if len(parts) > 3 else "10.0"
                
                # Reformat line: ID | ID | PHONEMES | DURATION
                fixed_lines.append(f"{file_id}\t{file_id}\t{phonemes}\t{duration}\n")
                
                # Create individual phoneme text file for each sample
                with open(os.path.join(phonemes_dir, f"{file_id}.txt"), "w", encoding="utf-8") as phn_f:
                    phn_f.write(phonemes)

    # Save fixed manifest back to disk
    with open(manifest_path, "w", encoding="utf-8") as f:
        f.writelines(fixed_lines)
    
    # Create validation manifest copy (required by the Trainer)
    val_manifest = manifest_path.replace("train.txt", "validation.txt")
    with open(val_manifest, "w", encoding="utf-8") as f:
        f.writelines(fixed_lines)

    print(f"✅ Manifest fixed and saved to {manifest_path}")
    print(f"✅ Phoneme files created in {phonemes_dir}")
else:
    print(f"❌ Error: Manifest not found at {manifest_path}")

Starting Manifest and Phonemes cleanup...
✅ Manifest fixed and saved to ./voicecraft_data/manifest/train.txt
✅ Phoneme files created in ./voicecraft_data/phonemes


In [7]:
# Directory containing the audio codes
encodec_dir = "./voicecraft_data/encodec_16khz_4codebooks"
converted_count = 0
renamed_count = 0

print("Starting Encodec files cleanup (Binary to Text)...")

if os.path.exists(encodec_dir):
    # Step A: Rename .codes files to .txt extension if needed
    for filename in os.listdir(encodec_dir):
        if filename.endswith(".codes"):
            old_path = os.path.join(encodec_dir, filename)
            new_path = os.path.join(encodec_dir, filename.replace(".codes", ".txt"))
            if not os.path.exists(new_path):
                os.rename(old_path, new_path)
                renamed_count += 1

    # Step B: Convert binary torch data to plain text numbers
    txt_files = [f for f in os.listdir(encodec_dir) if f.endswith(".txt")]
    for filename in txt_files:
        file_path = os.path.join(encodec_dir, filename)
        try:
            # Attempt to load as a torch binary file
            data = torch.load(file_path, map_location='cpu')
            
            # If successfully loaded as a tensor/object, convert to numpy
            if torch.is_tensor(data):
                data = data.numpy()
            
            # Format numbers as space-separated integers (required by VoiceCraft)
            if hasattr(data, 'tolist'):
                # One row per codebook
                lines = [" ".join(map(str, row)) for row in data]
                text_content = "\n".join(lines)
                
                # Overwrite binary file with plain text content
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(text_content)
                converted_count += 1
        except Exception:
            # If loading fails, the file is likely already in correct text format
            pass

    print(f"✅ Renamed {renamed_count} files to .txt")
    print(f"✅ Converted {converted_count} binary files to readable text.")
else:
    print(f"❌ Error: Encodec directory not found at {encodec_dir}")

Starting Encodec files cleanup (Binary to Text)...
✅ Renamed 0 files to .txt
✅ Converted 0 binary files to readable text.


In [ ]:
#5000 step - too many for fine tunning 
!torchrun --nproc_per_node=1 main.py \
--dataset_dir "./voicecraft_data" \
--exp_dir "./experiments/hebrew_fleurs/hebrew_v1_330M" \
--dataset "gigaspeech" \
--manifest_name "manifest" \
--text_vocab_size 46 \
--text_pad_token 46 \
--num_steps 5000 \
--lr 0.0005 \
--batch_size 4 \
--gradient_accumulation_steps 4 \
--n_codebooks 4 \
--audio_vocab_size 2048 \
--num_workers 4 \
--val_every_n_steps 500 \
--print_every_n_steps 10 



2026-02-27 17:57:33,245 [INFO] main.py:19 || Namespace(seed=1, precision='float16', num_workers=4, resume=False, tb_write_every_n_steps=100, print_every_n_steps=10, val_every_n_steps=500, lr=0.0005, batch_size=4, max_num_tokens=100000, val_max_num_tokens=None, num_buckets=6, dynamic_batching=0, weight_decay=0.01, warmup_fraction=0.01, num_epochs=10, num_steps=5000, gradient_accumulation_steps=4, gradient_clip_val=1.0, early_stop_step=3200, early_stop_threshold=-1.0, optimizer_name='AdamW', reduce_lr_start_step=3000, pseudo_epoch_size=3000, reduce_lr_start_epoch=4, clipping_update_period=600, exp_dir='./experiments/hebrew_fleurs/hebrew_v1_330M', dataset='gigaspeech', dataset_dir='./voicecraft_data', phn_folder_name='phonemes', encodec_folder_name='encodec_16khz_4codebooks', manifest_name='manifest', pad_x=1, audio_max_length=20, audio_min_length=2, text_max_length=400, text_min_length=10, encodec_sr=50, drop_long=0, mask_len_min=1, mask_len_max=600, eos=-1, reduced_eog=0, special_first=

In [1]:
# English comments: Checking the experiment directory for saved model checkpoints
import os

exp_dir = "./experiments/hebrew_fleurs/hebrew_v1_330M"
if os.path.exists(exp_dir):
    print(f"Contents of {exp_dir}:")
    files = os.listdir(exp_dir)
    for f in files:
        print(f" - {f}")
    
    # Specifically looking for .pth or .ckpt files
    checkpoints = [f for f in files if f.endswith('.pth') or f.endswith('.ckpt')]
    if checkpoints:
        print(f"\n✅ Found {len(checkpoints)} checkpoint(s)!")
    else:
        print("\nℹ️ No checkpoints found yet. The model probably didn't reach the saving interval.")
else:
    print(f"❌ Directory {exp_dir} not found.")

Contents of ./experiments/hebrew_fleurs/hebrew_v1_330M:
 - args.pkl
 - events.out.tfevents.1771832298.PF3NZKS1.87843.0
 - events.out.tfevents.1771710464.PF3NZKS1.66786.0
 - events.out.tfevents.1771711898.PF3NZKS1.72194.0
 - events.out.tfevents.1772207853.PF3NZKS1.23728.0
 - events.out.tfevents.1771711603.PF3NZKS1.70961.0
 - vocab.txt
 - events.out.tfevents.1771832456.PF3NZKS1.88356.0
 - events.out.tfevents.1771711548.PF3NZKS1.70703.0
 - events.out.tfevents.1771711844.PF3NZKS1.71916.0
 - events.out.tfevents.1771711113.PF3NZKS1.69093.0
 - events.out.tfevents.1771710544.PF3NZKS1.67142.0
 - events.out.tfevents.1771711290.PF3NZKS1.69872.0
 - events.out.tfevents.1771711213.PF3NZKS1.69474.0
 - events.out.tfevents.1771832832.PF3NZKS1.89567.0
 - events.out.tfevents.1772206867.PF3NZKS1.20167.0
 - events.out.tfevents.1771711721.PF3NZKS1.71460.0
 - events.out.tfevents.1771710963.PF3NZKS1.68510.0
 - events.out.tfevents.1771833070.PF3NZKS1.90427.0

ℹ️ No checkpoints found yet. The model probably did

In [2]:
# English comments: Counting unique phonemes across all text files to verify vocab size
import os

phn_dir = "./voicecraft_data/phonemes"
unique_phns = set()

for filename in os.listdir(phn_dir):
    if filename.endswith(".txt"):
        with open(os.path.join(phn_dir, filename), "r", encoding="utf-8") as f:
            # VoiceCraft phonemes are usually space-separated
            content = f.read().strip().split()
            unique_phns.update(content)

print(f"Total unique phonemes found: {len(unique_phns)}")
print(f"Recommended vocab size (unique + special tokens): {len(unique_phns) + 5}") # Adding a small buffer for PAD, BOS, EOS, etc.

Total unique phonemes found: 46
Recommended vocab size (unique + special tokens): 51


In [ ]:
# Export our new phoneme map to the format VoiceCraft expects (vocab.txt)
with open("vocab.txt", "w", encoding="utf-8") as f:
    # Sort by index to keep it organized
    sorted_vocab = sorted(new_phn2num.items(), key=lambda item: item[1])
    for phn, idx in sorted_vocab:
        f.write(f"{phn} {idx}\n")
print("Generated vocab.txt with 92 phonemes.")

In [3]:
# English comments: Locating and previewing all vocab.txt files to find the correct one
import os

# Search for all vocab.txt files in the current directory and subdirectories
found_files = []
for root, dirs, files in os.walk("."):
    for file in files:
        if file == "vocab.txt":
            full_path = os.path.join(root, file)
            found_files.append(full_path)

print(f"Found {len(found_files)} vocab files:\n")

for path in found_files:
    print(f"--- Checking: {path} ---")
    try:
        with open(path, "r", encoding="utf-8") as f:
            lines = f.readlines()
            print(f"Number of lines: {len(lines)}")
            print(f"First 5 lines: {[line.strip() for line in lines[:5]]}")
    except Exception as e:
        print(f"Could not read file: {e}")
    print("\n")

Found 3 vocab files:

--- Checking: ./vocab.txt ---
Number of lines: 92
First 5 lines: ['ɑː 0', 'u 1', 'aɪɚ 2', 'ɔ 3', 'x 4']


--- Checking: ./voicecraft_data/vocab.txt ---
Number of lines: 47
First 5 lines: ['0 !', '1 "', '2 (', '3 )', '4 ,']


--- Checking: ./experiments/hebrew_fleurs/hebrew_v1_330M/vocab.txt ---
Number of lines: 47
First 5 lines: ['0 !', '1 "', '2 (', '3 )', '4 ,']




In [ ]:
# missing parms
!torchrun --nproc_per_node=1 main.py \
--dataset_dir "./voicecraft_data" \
--exp_dir "./experiments/hebrew_fleurs/hebrew_v1_330M" \
--dataset "gigaspeech" \
--manifest_name "manifest" \
--text_vocab_size 46 \
--text_pad_token 46 \
--num_steps 5000 \
--lr 0.0005 \
--batch_size 4 \
--gradient_accumulation_steps 4 \
--n_codebooks 4 \
--audio_vocab_size 2048 \
--num_workers 4 \
--val_every_n_steps 500 \
--print_every_n_steps 10

2026-02-27 17:41:07,387 [INFO] main.py:19 || Namespace(seed=1, precision='float16', num_workers=4, resume=False, tb_write_every_n_steps=100, print_every_n_steps=10, val_every_n_steps=500, lr=0.0005, batch_size=4, max_num_tokens=100000, val_max_num_tokens=None, num_buckets=6, dynamic_batching=0, weight_decay=0.01, warmup_fraction=0.01, num_epochs=10, num_steps=5000, gradient_accumulation_steps=4, gradient_clip_val=1.0, early_stop_step=3200, early_stop_threshold=-1.0, optimizer_name='AdamW', reduce_lr_start_step=3000, pseudo_epoch_size=3000, reduce_lr_start_epoch=4, clipping_update_period=600, exp_dir='./experiments/hebrew_fleurs/hebrew_v1_330M', dataset='gigaspeech', dataset_dir='./voicecraft_data', phn_folder_name='phonemes', encodec_folder_name='encodec_16khz_4codebooks', manifest_name='manifest', pad_x=1, audio_max_length=20, audio_min_length=2, text_max_length=400, text_min_length=10, encodec_sr=50, drop_long=0, mask_len_min=1, mask_len_max=600, eos=-1, reduced_eog=0, special_first=

In [2]:
# Test if the tokenizer handles Hebrew

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model, tokenizer, and weights
# voicecraft_name = "giga330M.pth"
# ckpt_fn = f"./pretrained_models/{voicecraft_name}"
ckpt_fn = "pretrained_models/best_bundle_Enhanced.pth"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

ckpt = torch.load(ckpt_fn, map_location="cpu")
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
phn2num = ckpt['phn2num']

# Re-initialize the tokenizer with Hebrew support
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")
audio_tokenizer = AudioTokenizer(signature=encodec_fn) 

Dora directory: /tmp/audiocraft_sukiennik
/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [3]:
# hyperparameters for inference
codec_audio_sr = 16000
codec_sr = 50
top_k = 0
top_p = 0.5
# temperature = 0.8
temperature = 1
silence_tokens=[1388,1898,131]
kvcache = 1 # NOTE if OOM, change this to 0, or try the 330M model

# NOTE adjust the below three arguments if the generation is not as good
stop_repetition = 1 # NOTE if the model generate long silence, reduce the stop_repetition to 3, 2 or even 1
sample_batch_size = 5 # for gigaHalfLibri330M_TTSEnhanced_max16s.pth, 1 or 2 should be fine since the model is trained to do TTS, for the other two models, might need a higher number. NOTE: if the if there are long silence or unnaturally strecthed words, increase sample_batch_size to 5 or higher. What this will do to the model is that the model will run sample_batch_size examples of the same audio, and pick the one that's the shortest. So if the speech rate of the generated is too fast change it to a smaller number.
seed = 42 # change seed if you are still unhappy with the result

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed)

decode_config = {
    'top_k': top_k,
    'top_p': top_p,
    'temperature': temperature,
    'stop_repetition': stop_repetition,
    'kvcache': kvcache, 
    "codec_audio_sr": codec_audio_sr,
    "codec_sr": codec_sr, 
    "silence_tokens": silence_tokens, 
    "sample_batch_size": sample_batch_size}

In [5]:
# Running a zero-shot Hebrew inference test with the original 80-phoneme model

# Prepare the inputs for the first Hebrew generation
# Path to one of the 16kHz files we created
cut_off_sec = 4.3
audio_fn = "fleurs_hebrew/voicecraft_samples/test/sample_4330.wav"
# audio_fn = "demo/Daniella.wav"

# The transcript
# original_prompt_transcript = "זו הרכישה הגדולה ביותר בתולדות ebay"
# original_prompt_transcript = "טיולים הם פעילות מחוץ לבית, שכוללת הליכה בסביבה טבעית במקרים רבים בשבילי הליכה."

# The text you want the model to say in that voice
# target_transcript = "אני חושבת שתהיה תקיפה בחמישי בלילה"
# target_transcript_test = "ani khoshevet she-ti-ye tkifa be-khamishi ba-layla"
# target_transcript_test = "זו הרכישה הגדולה ביותר בתולדות "
# target_transcript_test = "This is a test to see if you can speak."
# target_transcript_test = "This is a longer test to ensure the model has enough audio data to decode properly without crashing."
# target_transcript_test = "Hello, my name is Daniella. I am currently exploring new technologies and testing how amazing this model is, i can't beleive it works"
# target_transcript = "טיולים הם פעילות מחוץ לבית, שכוללת הליכה בסביבה טבעית אני חושבת שתהיה תקיפה בחמישי בלילה"
# target_transcript = "גם גולף וגם רוגבי אמורים לחזור למשחקים האולימפיים"
# target_transcript = "במהלך השנים למדתי ועבדתי"
target_transcript =  "להקות אריות מתנהגות באופן דומה מאוד ללהקות זאבים או חתולים חיות הדומות בהתנהגותן באופן לא מפתיע"
# Get frame count
info = torchaudio.info(audio_fn)
# prompt_end_frame = info.num_frames 
prompt_end_frame = int(cut_off_sec * info.sample_rate)

# Run Inference
with torch.no_grad():
    with torch.cuda.amp.autocast():
        # NOTE: Using target_transcript (Hebrew) and text_tokenizer_he
        concated_audio, gen_audio = inference_one_sample(
            model, 
            ckpt["config"], 
            phn2num,                # This is the 80-phoneme map from the model
            # text_tokenizer,  
            text_tokenizer_he,      # Hebrew G2P
            audio_tokenizer, 
            audio_fn, 
            target_transcript,     # The actual Hebrew text
            # target_transcript_test, 
            device, 
            decode_config, 
            prompt_end_frame
        )

# Post-processing for display
concated_audio, gen_audio = concated_audio[0].cpu(), gen_audio[0].cpu()

# display the audio
from IPython.display import Audio
print("Concatenate prompt and generated:")
display(Audio(concated_audio, rate=codec_audio_sr))

print("Generated Audio (Zero-shot Hebrew):")
display(Audio(gen_audio, rate=codec_audio_sr))

/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv1d(input, weight, bias, self.stride,


Concatenate prompt and generated:


Generated Audio (Zero-shot Hebrew):


In [4]:
# Running a zero-shot Hebrew inference test with the original 80-phoneme model

# Prepare the inputs for the first Hebrew generation
# Path to one of the 16kHz files we created
cut_off_sec = 6.1
audio_fn = "fleurs_hebrew/voicecraft_samples/test/sample_4346.wav"
# audio_fn = "demo/Daniella.wav"

# The transcript
# original_prompt_transcript = "זו הרכישה הגדולה ביותר בתולדות ebay"
# original_prompt_transcript = "טיולים הם פעילות מחוץ לבית, שכוללת הליכה בסביבה טבעית במקרים רבים בשבילי הליכה."

target_transcript =  "בתוך שנתיים מסיום המלחמה בעלות הברית לשעבר הפכו לאויבות והמלחמה הקרה החלה ונמשכה שנים רבות ופגעה בכלכלה"
# Get frame count
info = torchaudio.info(audio_fn)
# prompt_end_frame = info.num_frames 
prompt_end_frame = int(cut_off_sec * info.sample_rate)

# Run Inference
with torch.no_grad():
    with torch.cuda.amp.autocast():
        # NOTE: Using target_transcript (Hebrew) and text_tokenizer_he
        concated_audio, gen_audio = inference_one_sample(
            model, 
            ckpt["config"], 
            phn2num,                # This is the 80-phoneme map from the model
            # text_tokenizer,  
            text_tokenizer_he,      # Hebrew G2P
            audio_tokenizer, 
            audio_fn, 
            target_transcript,     # The actual Hebrew text
            # target_transcript_test, 
            device, 
            decode_config, 
            prompt_end_frame
        )

# Post-processing for display
concated_audio, gen_audio = concated_audio[0].cpu(), gen_audio[0].cpu()

# display the audio
from IPython.display import Audio
print("Concatenate prompt and generated:")
display(Audio(concated_audio, rate=codec_audio_sr))

print("Generated Audio (Zero-shot Hebrew):")
display(Audio(gen_audio, rate=codec_audio_sr))

/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv1d(input, weight, bias, self.stride,


Concatenate prompt and generated:


Generated Audio (Zero-shot Hebrew):


With nikod

In [2]:
# Test if the tokenizer handles Hebrew

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model, tokenizer, and weights
# voicecraft_name = "giga330M.pth"
# ckpt_fn = f"./pretrained_models/{voicecraft_name}"
ckpt_fn = "pretrained_models/best_bundle_nikod.pth"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

ckpt = torch.load(ckpt_fn, map_location="cpu")
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
phn2num = ckpt['phn2num']

# Re-initialize the tokenizer with Hebrew support
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")
audio_tokenizer = AudioTokenizer(signature=encodec_fn) 

Dora directory: /tmp/audiocraft_sukiennik
/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [11]:
# hyperparameters for inference
codec_audio_sr = 16000
codec_sr = 50
top_k = 0
top_p = 0.5
# temperature = 0.8
temperature = 1
silence_tokens=[1388,1898,131]
kvcache = 1 # NOTE if OOM, change this to 0, or try the 330M model

# NOTE adjust the below three arguments if the generation is not as good
stop_repetition = 3 # NOTE if the model generate long silence, reduce the stop_repetition to 3, 2 or even 1
sample_batch_size = 3 # for gigaHalfLibri330M_TTSEnhanced_max16s.pth, 1 or 2 should be fine since the model is trained to do TTS, for the other two models, might need a higher number. NOTE: if the if there are long silence or unnaturally strecthed words, increase sample_batch_size to 5 or higher. What this will do to the model is that the model will run sample_batch_size examples of the same audio, and pick the one that's the shortest. So if the speech rate of the generated is too fast change it to a smaller number.
seed = 42 # change seed if you are still unhappy with the result

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed)

decode_config = {
    'top_k': top_k,
    'top_p': top_p,
    'temperature': temperature,
    'stop_repetition': stop_repetition,
    'kvcache': kvcache, 
    "codec_audio_sr": codec_audio_sr,
    "codec_sr": codec_sr, 
    "silence_tokens": silence_tokens, 
    "sample_batch_size": sample_batch_size}

In [11]:
def fix_for_inference(text):
    if not text: return ""
    text = re.sub(r'([^ו])ֹ', r'\1וֹ', text) # חולם חסר
    text = text.replace("לֹא", "לוֹא").replace('ּ', '') # לא ודגשים
    text = text.replace('ׁ', '').replace('ׂ', '') # נקודות ש"ין ושׂ"ין
    return text

# 2. תנקי את הטקסט שלך
# target_transcript = "בְּתוֹךְ שְׁנָתַיִים מִסִּיּוּם הַמִּלְחָמָה בַּעֲלוֹת הַבְּרִית לְשֶׁעָבַר הָפְכוּ לָאוֹיְבוֹת וְהַמִּלְחָמָה הַקָּרָה הֵחֵלָּה וְנִמְשְׁכָה שָׁנִים רַבּוֹת וּפָגְעָה בְּכַלְכָּלָה"
# target_transcript = "שָׁלוֹם קוֹרְאִים לִי דָּנִיאֵלָה וַאֲנִי סְטוּדֶנְטִית לְתוֹאַר שֵׁנִי בְּהַנְדָּסַת חַשְׁמַל וְהַמּוֹדֵל הַזֶּה מְיַיצֵּר מִילִּים לְבַד"
target_transcript = "שָׁלוֹם קוֹרְאִים לִי דָּנִיאֵלָה וַאֲנִי סְטוּדֶנְטִית לְתוֹאַר שֵׁנִי בְּהַנְדָּסַת חַשְׁמַל בַּקּוּרְס הַזֶּה אֲנִי בּוֹדֶקֶת אֵיךְ הַמּוֹדֵל הַזֶּה מְיַיצֵּר מִילִּים לְבַד"


clean_target = fix_for_inference(target_transcript)

In [12]:
# Running a zero-shot Hebrew inference test with the original 80-phoneme model

# Prepare the inputs for the first Hebrew generation
# Path to one of the 16kHz files we created
cut_off_sec = 6.1
audio_fn = "fleurs_hebrew/voicecraft_samples/test/sample_4346.wav"
# audio_fn = "demo/Daniella.wav"

# The transcript
# original_prompt_transcript = "זו הרכישה הגדולה ביותר בתולדות ebay"
# original_prompt_transcript = "טיולים הם פעילות מחוץ לבית, שכוללת הליכה בסביבה טבעית במקרים רבים בשבילי הליכה."

# target_transcript =  "בתוך שנתיים מסיום המלחמה בעלות הברית לשעבר הפכו לאויבות והמלחמה הקרה החלה ונמשכה שנים רבות ופגעה בכלכלה"
target_transcript = "בְּתוֹךְ שְׁנָתַיִים מִסִּיּוּם הַמִּלְחָמָה בַּעֲלוֹת הַבְּרִית לְשֶׁעָבַר הָפְכוּ לָאוֹיְבוֹת וְהַמִּלְחָמָה הַקָּרָה הֵחֵלָּה וְנִמְשְׁכָה שָׁנִים רַבּוֹת וּפָגְעָה בְּכַלְכָּלָה"

# Get frame count
info = torchaudio.info(audio_fn)
# prompt_end_frame = info.num_frames 
prompt_end_frame = int(cut_off_sec * info.sample_rate)

# Run Inference
with torch.no_grad():
    with torch.cuda.amp.autocast():
        # NOTE: Using target_transcript (Hebrew) and text_tokenizer_he
        concated_audio, gen_audio = inference_one_sample(
            model, 
            ckpt["config"], 
            phn2num,                # This is the 80-phoneme map from the model
            # text_tokenizer,  
            text_tokenizer_he,      # Hebrew G2P
            audio_tokenizer, 
            audio_fn, 
            clean_target,     # The actual Hebrew text
            # target_transcript_test, 
            device, 
            decode_config, 
            prompt_end_frame
        )

# Post-processing for display
concated_audio, gen_audio = concated_audio[0].cpu(), gen_audio[0].cpu()

# display the audio
from IPython.display import Audio
print("Concatenate prompt and generated:")
display(Audio(concated_audio, rate=codec_audio_sr))

print("Generated Audio (Zero-shot Hebrew):")
display(Audio(gen_audio, rate=codec_audio_sr))

Concatenate prompt and generated:


Generated Audio (Zero-shot Hebrew):


In [12]:
# Test if the tokenizer handles Hebrew

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model, tokenizer, and weights
# voicecraft_name = "giga330M.pth"
# ckpt_fn = f"./pretrained_models/{voicecraft_name}"
# ckpt_fn = "pretrained_models/best_bundle_Enhanced.pth"
ckpt_fn = "pretrained_models/best_bundle_merged_nikod_4.pth"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

ckpt = torch.load(ckpt_fn, map_location="cpu")
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
phn2num = ckpt['phn2num']

# Re-initialize the tokenizer with Hebrew support
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")
audio_tokenizer = AudioTokenizer(signature=encodec_fn) 

/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [15]:
# hyperparameters for inference
codec_audio_sr = 16000
codec_sr = 50
top_k = 0
top_p = 0.5
temperature = 0.8
# temperature = 1
silence_tokens=[1388,1898,131]
kvcache = 1 # NOTE if OOM, change this to 0, or try the 330M model

# NOTE adjust the below three arguments if the generation is not as good
stop_repetition = 1 # NOTE if the model generate long silence, reduce the stop_repetition to 3, 2 or even 1
sample_batch_size = 1 # for gigaHalfLibri330M_TTSEnhanced_max16s.pth, 1 or 2 should be fine since the model is trained to do TTS, for the other two models, might need a higher number. NOTE: if the if there are long silence or unnaturally strecthed words, increase sample_batch_size to 5 or higher. What this will do to the model is that the model will run sample_batch_size examples of the same audio, and pick the one that's the shortest. So if the speech rate of the generated is too fast change it to a smaller number.
seed = 42 # change seed if you are still unhappy with the result

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed)

decode_config = {
    'top_k': top_k,
    'top_p': top_p,
    'temperature': temperature,
    'stop_repetition': stop_repetition,
    'kvcache': kvcache, 
    "codec_audio_sr": codec_audio_sr,
    "codec_sr": codec_sr, 
    "silence_tokens": silence_tokens, 
    "sample_batch_size": sample_batch_size}

In [16]:
# Running a zero-shot Hebrew inference test with the original 80-phoneme model

# Prepare the inputs for the first Hebrew generation
# Path to one of the 16kHz files we created
cut_off_sec = 6
# audio_fn = "fleurs_hebrew/voicecraft_samples/test/sample_4346.wav"
audio_fn = "demo/Daniella2Heb.wav"

# The transcript
# original_prompt_transcript = "זו הרכישה הגדולה ביותר בתולדות ebay"
# original_prompt_transcript = "טיולים הם פעילות מחוץ לבית, שכוללת הליכה בסביבה טבעית במקרים רבים בשבילי הליכה."

# target_transcript =  "בתוך שנתיים מסיום המלחמה בעלות הברית לשעבר הפכו לאויבות והמלחמה הקרה החלה ונמשכה שנים רבות ופגעה בכלכלה"
# target_transcript = "בְּתוֹךְ שְׁנָתַיִים מִסִּיּוּם הַמִּלְחָמָה בַּעֲלוֹת הַבְּרִית לְשֶׁעָבַר הָפְכוּ לָאוֹיְבוֹת וְהַמִּלְחָמָה הַקָּרָה הֵחֵלָּה וְנִמְשְׁכָה שָׁנִים רַבּוֹת וּפָגְעָה בְּכַלְכָּלָה"
# target_transcript = "שלום קוראים לי דניאלה ואני סטודנטית לתואר שני בהנדסת חשמל בקורס הזה אני בודקת איך המודל הזה מייצר מילים לבד"
# target_transcript = "שָׁלוֹם קוֹרְאִים לִי דָּנִיאֵלָה וַאֲנִי סְטוּדֶנְטִית לְתוֹאַר שֵׁנִי בְּהַנְדָּסַת חַשְׁמַל וְהַמּוֹדֵל הַזֶּה מְיַיצֵּר מִילִּים לְבַד"
target_transcript = "שָׁלוֹם קוֹרְאִים לִי דָּנִיאֵלָה וַאֲנִי סְטוּדֶנְטִית לְתוֹאַר שֵׁנִי בְּהַנְדָּסַת חַשְׁמַל בַּקּוּרְס הַזֶּה אֲנִי בּוֹדֶקֶת אֵיךְ הַמּוֹדֵל הַזֶּה מְיַיצֵּר מִילִּים לְבַד"


# Get frame count
info = torchaudio.info(audio_fn)
# prompt_end_frame = info.num_frames 
prompt_end_frame = int(cut_off_sec * info.sample_rate)

# Run Inference
with torch.no_grad():
    with torch.cuda.amp.autocast():
        # NOTE: Using target_transcript (Hebrew) and text_tokenizer_he
        concated_audio, gen_audio = inference_one_sample(
            model, 
            ckpt["config"], 
            phn2num,                # This is the 80-phoneme map from the model
            # text_tokenizer,  
            text_tokenizer_he,      # Hebrew G2P
            audio_tokenizer, 
            audio_fn, 
            clean_target,     # The actual Hebrew text
            # target_transcript_test, 
            device, 
            decode_config, 
            prompt_end_frame
        )

# Post-processing for display
concated_audio, gen_audio = concated_audio[0].cpu(), gen_audio[0].cpu()

# display the audio
from IPython.display import Audio
print("Concatenate prompt and generated:")
display(Audio(concated_audio, rate=codec_audio_sr))

print("Generated Audio (Zero-shot Hebrew):")
display(Audio(gen_audio, rate=codec_audio_sr))

Concatenate prompt and generated:


Generated Audio (Zero-shot Hebrew):


In [18]:
# Test if the tokenizer handles Hebrew

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model, tokenizer, and weights
# voicecraft_name = "giga330M.pth"
# ckpt_fn = f"./pretrained_models/{voicecraft_name}"
# ckpt_fn = "pretrained_models/best_bundle_Enhanced.pth"
ckpt_fn = "pretrained_models/best_bundle_merge_nikod_19.pth"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

ckpt = torch.load(ckpt_fn, map_location="cpu")
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
phn2num = ckpt['phn2num']

# Re-initialize the tokenizer with Hebrew support
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")
audio_tokenizer = AudioTokenizer(signature=encodec_fn) 

In [19]:
def fix_for_inference(text):
    if not text: return ""
    text = re.sub(r'([^ו])ֹ', r'\1וֹ', text) # חולם חסר
    text = text.replace("לֹא", "לוֹא").replace('ּ', '') # לא ודגשים
    text = text.replace('ׁ', '').replace('ׂ', '') # נקודות ש"ין ושׂ"ין
    return text

# 2. תנקי את הטקסט שלך
# target_transcript = "בְּתוֹךְ שְׁנָתַיִים מִסִּיּוּם הַמִּלְחָמָה בַּעֲלוֹת הַבְּרִית לְשֶׁעָבַר הָפְכוּ לָאוֹיְבוֹת וְהַמִּלְחָמָה הַקָּרָה הֵחֵלָּה וְנִמְשְׁכָה שָׁנִים רַבּוֹת וּפָגְעָה בְּכַלְכָּלָה"
# target_transcript = "שָׁלוֹם קוֹרְאִים לִי דָּנִיאֵלָה וַאֲנִי סְטוּדֶנְטִית לְתוֹאַר שֵׁנִי בְּהַנְדָּסַת חַשְׁמַל וְהַמּוֹדֵל הַזֶּה מְיַיצֵּר מִילִּים לְבַד"
target_transcript = "שָׁלוֹם קוֹרְאִים לִי דָּנִיאֵלָה וַאֲנִי סְטוּדֶנְטִית לְתוֹאַר שֵׁנִי בְּהַנְדָּסַת חַשְׁמַל בַּקּוּרְס הַזֶּה אֲנִי בּוֹדֶקֶת אֵיךְ הַמּוֹדֵל הַזֶּה מְיַיצֵּר מִילִּים לְבַד"


clean_target = fix_for_inference(target_transcript)

In [20]:
# hyperparameters for inference
codec_audio_sr = 16000
codec_sr = 50
top_k = 0
top_p = 0.5
temperature = 0.8
# temperature = 1
silence_tokens=[1388,1898,131]
kvcache = 1 # NOTE if OOM, change this to 0, or try the 330M model

# NOTE adjust the below three arguments if the generation is not as good
stop_repetition = 1 # NOTE if the model generate long silence, reduce the stop_repetition to 3, 2 or even 1
sample_batch_size = 1 # for gigaHalfLibri330M_TTSEnhanced_max16s.pth, 1 or 2 should be fine since the model is trained to do TTS, for the other two models, might need a higher number. NOTE: if the if there are long silence or unnaturally strecthed words, increase sample_batch_size to 5 or higher. What this will do to the model is that the model will run sample_batch_size examples of the same audio, and pick the one that's the shortest. So if the speech rate of the generated is too fast change it to a smaller number.
seed = 42 # change seed if you are still unhappy with the result

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed)

decode_config = {
    'top_k': top_k,
    'top_p': top_p,
    'temperature': temperature,
    'stop_repetition': stop_repetition,
    'kvcache': kvcache, 
    "codec_audio_sr": codec_audio_sr,
    "codec_sr": codec_sr, 
    "silence_tokens": silence_tokens, 
    "sample_batch_size": sample_batch_size}

In [21]:
# Running a zero-shot Hebrew inference test with the original 80-phoneme model

# Prepare the inputs for the first Hebrew generation
# Path to one of the 16kHz files we created
cut_off_sec = 6
# audio_fn = "fleurs_hebrew/voicecraft_samples/test/sample_4346.wav"
audio_fn = "demo/Daniella2Heb.wav"

# The transcript
# original_prompt_transcript = "זו הרכישה הגדולה ביותר בתולדות ebay"
# original_prompt_transcript = "טיולים הם פעילות מחוץ לבית, שכוללת הליכה בסביבה טבעית במקרים רבים בשבילי הליכה."

# target_transcript =  "בתוך שנתיים מסיום המלחמה בעלות הברית לשעבר הפכו לאויבות והמלחמה הקרה החלה ונמשכה שנים רבות ופגעה בכלכלה"
# target_transcript = "בְּתוֹךְ שְׁנָתַיִים מִסִּיּוּם הַמִּלְחָמָה בַּעֲלוֹת הַבְּרִית לְשֶׁעָבַר הָפְכוּ לָאוֹיְבוֹת וְהַמִּלְחָמָה הַקָּרָה הֵחֵלָּה וְנִמְשְׁכָה שָׁנִים רַבּוֹת וּפָגְעָה בְּכַלְכָּלָה"
# target_transcript = "שלום קוראים לי דניאלה ואני סטודנטית לתואר שני בהנדסת חשמל בקורס הזה אני בודקת איך המודל הזה מייצר מילים לבד"
# target_transcript = "שָׁלוֹם קוֹרְאִים לִי דָּנִיאֵלָה וַאֲנִי סְטוּדֶנְטִית לְתוֹאַר שֵׁנִי בְּהַנְדָּסַת חַשְׁמַל וְהַמּוֹדֵל הַזֶּה מְיַיצֵּר מִילִּים לְבַד"
target_transcript = "שָׁלוֹם קוֹרְאִים לִי דָּנִיאֵלָה וַאֲנִי סְטוּדֶנְטִית לְתוֹאַר שֵׁנִי בְּהַנְדָּסַת חַשְׁמַל בַּקּוּרְס הַזֶּה אֲנִי בּוֹדֶקֶת אֵיךְ הַמּוֹדֵל הַזֶּה מְיַיצֵּר מִילִּים לְבַד"


# Get frame count
info = torchaudio.info(audio_fn)
# prompt_end_frame = info.num_frames 
prompt_end_frame = int(cut_off_sec * info.sample_rate)

# Run Inference
with torch.no_grad():
    with torch.cuda.amp.autocast():
        # NOTE: Using target_transcript (Hebrew) and text_tokenizer_he
        concated_audio, gen_audio = inference_one_sample(
            model, 
            ckpt["config"], 
            phn2num,                # This is the 80-phoneme map from the model
            # text_tokenizer,  
            text_tokenizer_he,      # Hebrew G2P
            audio_tokenizer, 
            audio_fn, 
            clean_target,     # The actual Hebrew text
            # target_transcript_test, 
            device, 
            decode_config, 
            prompt_end_frame
        )

# Post-processing for display
concated_audio, gen_audio = concated_audio[0].cpu(), gen_audio[0].cpu()

# display the audio
from IPython.display import Audio
print("Concatenate prompt and generated:")
display(Audio(concated_audio, rate=codec_audio_sr))

print("Generated Audio (Zero-shot Hebrew):")
display(Audio(gen_audio, rate=codec_audio_sr))

Concatenate prompt and generated:


Generated Audio (Zero-shot Hebrew):
